  1. Load a representative subset of documents (e.g., 50–100 docs across diverse sources: Slack, Gmail, Jira, Confluence) and questions from EnterpriseRAG-Bench.                         
  2. Phase 1 (Data Cleaning & Normalization):                                                                                                                                             
      • Unicode NFKC normalization and whitespace cleanup.                                                                                                                                
      • Source-specific cleaning (e.g., Slack entity tag handling, email quote stripping).                                                                                                
      • Exact deduplication (SHA-256) check.                                                                                                                                              
  3. Phase 2 (Chunking):                                                                                                                                                                  
      • Parent-child hierarchical split (or recursive chunking) attaching metadata (doc_id, source_type).                                                                                 
  4. Phase 3 & 5 (Indexing & Hybrid Retrieval):                                                                                                                                           
      • Dense vector embeddings + Sparse BM25 index.                                                                                                                                      
      • Hybrid search test combining both with RRF.                                                                                                                                       
  5. Phase 6 (Re-Ranking):                                                                                                                                                                
      • Scoring top candidates using a cross-encoder model.                                                                                                                               
  6. Phase 7 (Generation & Prompting):                                                                                                                                                    
      • Formatting context with citations ([Doc ID]) and testing the output with an LLM.                                                                                                  
                                                                                                 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv('/home/mahavir/code/python_projects/rag_benchmark/.env.dev')

hugguing_face_token = os.getenv('HUGGINGFACE_TOKEN')


# Phase 1. Data Loading

In [2]:
from datasets import load_dataset
DATASET_NAME = "ingalepratap/EnterpriseRAG-Bench"

/home/mahavir/code/python_projects/rag_benchmark/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
raw_dataset_questions = load_dataset(DATASET_NAME,'questions')
raw_dataset_questions

DatasetDict({
    test: Dataset({
        features: ['question_id', 'question_type', 'source_types', 'question', 'expected_doc_ids', 'gold_answer', 'answer_facts'],
        num_rows: 500
    })
})

In [4]:
raw_dataset_documents = load_dataset(DATASET_NAME,'documents')
raw_dataset_documents

DatasetDict({
    test: Dataset({
        features: ['doc_id', 'source_type', 'title', 'content'],
        num_rows: 511962
    })
})

Step 1: Data Cleaning ( Basic Cleaning & Normalization)

In [ ]:
import unicodedata
import re

In [ ]:
def normalize_text_for_rag(text:str)->str:
    
    # unicode NFKC Normalization (fixes ligatures,full width charas,accents)
    text = unicodedata.normalize("NFKC",text)

    # unify line breaks to standard unix format(\r\r and \r->\n)
    text = text.replace('\r\n','\n').replace("\r", "\n")

    # replace non breaking space and zero width spaces with stanard space
    # \u00a0 = non-breaking space, \u200b = zero-width space, \u202f = narrow no-break space
    text = re.sub(r"[\u00a0\u2007\xa0]"," ",text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)  # Strip zero-width markers

    # Step 4: Stitch hyphenated words broken across line wraps
    # Example: 'multi-\nlayer' -> 'multilayer'
    text = re.sub(r"(\b[a-zA-Z]+)-\n([a-zA-Z]+\b)", r"\1\2", text)

    # Step 5: Strip unprintable control characters (keep \n and \t)
    # Regex keeps standard ASCII, unicode letters/numbers/symbols, and basic whitespace
    text = "".join(
        ch for ch in text 
        if unicodedata.category(ch)[0] != "C" or ch in ("\n", "\t")
    )

    # Step 6: Normalize horizontal whitespace (spaces and tabs -> single space)
    text = re.sub(r"[ \t]+", " ", text)

    # Step 7: Collapse redundant blank lines (allow max 2 newlines for paragraph boundaries)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Step 8: Strip leading and trailing whitespace per line
    text = "\n".join(line.strip() for line in text.split("\n"))

    return text.strip()


In [ ]:
sample_raw_text = (
    "The ﬁnancial state\u00a0ment for Q3 shows a 25% in-\n"
    "crease in revenue.\r\n\r\n\r\n"
    "Full-width text: Ｔｅｓｔ Ｃａｓｅ.\n"
    "Ligatures test: ﬂow, aﬃliate.\n"
    "Erratic    spacing\t\tand   tabs."
)

In [ ]:
raw_dataset_documents['test'][0]

In [ ]:
raw_dataset_questions['test'][0]

In [ ]:
raw_dataset_documents['test'][0]

In [ ]:
sample_q = raw_dataset_questions['test'][0]
normalize_sample_q = normalize_text_for_rag(sample_q['question'])
normalize_sample_q

In [ ]:
sample_d = raw_dataset_documents['test'][0]
normalize_sample_d_title = normalize_text_for_rag(sample_d['title'])
normalize_sample_d_content = normalize_text_for_rag(sample_d['content'])


In [ ]:
def process_batch_questions(batch):
    return {'normalized_question': [normalize_text_for_rag(q) for q in batch["question"]]}

In [ ]:
raw_dataset_questions= raw_dataset_questions.map(process_batch_questions,batched=True,num_proc=12)


In [ ]:
raw_dataset_questions['test'][0]

In [ ]:
raw_dataset_documents['test'][0]

In [ ]:
def process_batch_documents(batch):
    batch['normalized_title'] = [ normalize_text_for_rag(t) for t in batch["title"]]
    batch['normalized_content'] = [ normalize_text_for_rag(t) for t in batch["content"]]
    return batch

In [ ]:
raw_dataset_documents = raw_dataset_documents.map(process_batch_documents,batched=True,num_proc=12)

In [ ]:
raw_dataset_documents['test'][0]

Step 2 Exact De Duplicaiton (SHA256)

In [ ]:
import hashlib

In [ ]:
def compute_sha256(text:str)->str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

In [ ]:
seen_hashes = {}
duplicate_doc_ids = []

for doc in raw_dataset_documents['test']:
    doc_id = doc['doc_id']
    normalized_content = doc['normalized_content']
    content_hash = compute_sha256(normalized_content)

    if content_hash in seen_hashes:
        duplicate_doc_ids.append((doc_id,seen_hashes[content_hash]))
    else:
        seen_hashes[content_hash] = doc_id

In [ ]:
print(f"Total documents: {len(raw_dataset_documents['test'])}")                                                                                                                       
print(f"Unique documents: {len(seen_hashes)}")                                                                                                                                        
print(f"Duplicate documents found: {len(duplicate_doc_ids)}")     

Step 3: Source-Specific Inspection & Cleaning     

In [ ]:
source_types = set(raw_dataset_documents['test']['source_type'])                                                                                                                      
print("Available source types:", source_types)                                                                                                                                        
                                                                                                                                                                                          
# Inspect 1 sample from each source type                                                                                                                                              
samples_by_source = {}                                                                                                                                                                
for doc in raw_dataset_documents['test']:                                                                                                                                             
    st = doc['source_type']                                                                                                                                                           
    if st not in samples_by_source:                                                                                                                                                   
        samples_by_source[st] = doc                                                                                                                                                   
    if len(samples_by_source) == len(source_types):                                                                                                                                   
        break                                                                                                                                                                         
                                                                                                                                                                                          
for st, doc in samples_by_source.items():                                                                                                                                             
    print(f"\n=================== SOURCE: {st.upper()} ===================")                                                                                                          
    print("Title:", doc['title'])                                                                                                                                                     
    print("Content preview (first 250 chars):\n", doc['normalized_content'][:250])                                                                                                    
                                                                                                      

In [ ]:
samples_by_source

In [ ]:
seen_hashes

In [ ]:
duplicate_doc_ids

In [ ]:
seen_hashes

# Phase 2 (Chunking)

In [ ]:
import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
subset_docs = raw_dataset_documents['test'].select(range(1000))

In [ ]:
# 1. Define Splitters
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=300,
    separators=["\n\n", "\n", ". ", " "]
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "]
)

# 2. Build Parent-Child Hierarchy with Enterprise Metadata Lineage
parent_store = {}  # Acts as the raw document store (e.g., Redis / MongoDB)
child_records = [] # Passed to Vector DB + BM25

for doc in subset_docs:
    doc_id = doc["doc_id"]
    source_type = doc["source_type"]
    title = doc.get("normalized_title", "")
    full_text = doc["normalized_content"]
    
    if not full_text.strip():
        continue

    # Generate Parent Chunks
    parents = parent_splitter.split_text(full_text)
    
    for p_idx, parent_text in enumerate(parents):
        parent_id = f"{doc_id}_p{p_idx}"
        
        # Save parent text with structural metadata
        parent_store[parent_id] = {
            "parent_id": parent_id,
            "doc_id": doc_id,
            "source_type": source_type,
            "title": title,
            "parent_text": parent_text
        }
        
        # Split parent into granular child chunks
        children = child_splitter.split_text(parent_text)
        for c_idx, child_text in enumerate(children):
            child_id = f"{parent_id}_c{c_idx}"
            child_records.append({
                "child_id": child_id,
                "parent_id": parent_id,
                "doc_id": doc_id,
                "source_type": source_type,
                "title": title,
                "child_text": child_text
            })

print(f"Ingested {len(subset_docs)} documents -> Generated {len(parent_store)} Parents, {len(child_records)} Child chunks.")

In [ ]:
parent_store.keys()

In [ ]:
parent_store['dsid_e54ef48bae78474684a957cf613d47d5_p0']

In [ ]:
child_records

# Phase 3 - Indexing and Storing

In [ ]:
child_records[0]

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [record['child_text'].lower().split() for record in child_records]
bm25_index = BM25Okapi(tokenized_corpus)

print(f"BM25 Index built over {len(tokenized_corpus)} chunks.") 

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
child_texts = [r["child_text"] for r in child_records]

In [ ]:
child_embeddings = embedding_model.encode(
    child_texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

In [ ]:
child_embeddings.shape

In [ ]:
test_query = "perf-canary rollback procedure"                                                                                                                                         
                                                                                                                                                                                          
# 2. Tokenize the query exactly like the corpus (lowercase + split)                                                                                                                   
tokenized_query = test_query.lower().split()                                                                                                                                          
                                                                                                                                                                                          
# 3. Get BM25 scores across all 17,771 chunks                                                                                                                                         
bm25_scores = bm25_index.get_scores(tokenized_query)                                                                                                                                  
                                                                                                                                                                                          
# 4. Get the Top 3 highest scoring chunk indices                                                                                                                                      
top_indices = np.argsort(bm25_scores)[::-1][:3]                                                                                                                                       
                                                                                                                                                                                          
print(f"=== TOP 3 BM25 HITS FOR: '{test_query}' ===")                                                                                                                                 
for rank, idx in enumerate(top_indices, 1):                                                                                                                                           
    chunk = child_records[idx]                                                                                                                                                        
    print(f"\n[Rank {rank}] Score: {bm25_scores[idx]:.4f} | Source: {chunk['source_type'].upper()}")                                                                                  
    print(f"Title: {chunk['title']}")                                                                                                                                                 
    print(f"Parent ID: {chunk['parent_id']}")                                                                                                                                         
    print(f"Content: {chunk['child_text'][:250]}...")   

# Phase 4 - Retrivel(hubrid Search)

In [ ]:
def hybrid_search(query:str,top_k:int = 50,rrf_k=60,candidate_pool:int = 50):

    # Step 1: Sparse Search
    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    top_bm25_indices = np.argsort(bm25_scores)[::-1][:candidate_pool]

    # Step 2: Dense search 
    query_vector = embedding_model.encode(query,normalize_embeddings=True)
    dense_scores = child_embeddings @  query_vector
    top_dense_indices = np.argsort(dense_scores)[::-1][:candidate_pool]

    # Calculate Reciprocal Rank Fusion(RRF)
    rrf_scores = {}

    for rank,idx in enumerate(top_bm25_indices,start=1):
        rrf_scores[idx] = rrf_scores.get(idx,0.0) + (1.0 / (rrf_k + rank))

    for rank,idx in enumerate(top_dense_indices,start=1):
        rrf_scores[idx] = rrf_scores.get(idx,0.0) + (1.0 / (rrf_k + rank))

    # sort top candidates by combined RRF Scores
    fused_indices = sorted(rrf_scores.keys(),key=lambda i: rrf_scores[i],reverse=True)[:top_k]

    #Parent COntext Hydration
    hydrated_results = []  
    for rank,idx in enumerate(fused_indices,start=1):
        child = child_records[idx]
        parent_id = child['parent_id']
        parent = parent_store[parent_id]

        hydrated_results.append({                                                                                                                                                     
                "rank": rank,                                                                                                                                                             
                "rrf_score": rrf_scores[idx],                                                                                                                                             
                "bm25_score": float(bm25_scores[idx]),                                                                                                                                    
                "dense_score": float(dense_scores[idx]),                                                                                                                                  
                "child_id": child["child_id"],                                                                                                                                            
                "parent_id": parent_id,                                                                                                                                                   
                "doc_id": child["doc_id"],                                                                                                                                                
                "source_type": child["source_type"],                                                                                                                                      
                "title": child["title"],
                "matched_child_text": child["child_text"],
                "hydrated_parent_text": parent["parent_text"]
            })
    return hydrated_results


In [ ]:
# test query

test_query= "What is the procedure to revert a canary deployment if it breaks?"

results = hybrid_search(test_query)
results

In [ ]:
len(results)

In [ ]:
results[0]

# Phase 5 - Re Ranking

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker

In [ ]:
def rerank_results(query: str, retrieved_candidates: list[dict], top_n: int = 3) -> list[dict]:                                                                                       
      """                                                                                                                                                                               
      Takes the Top 50 candidates from Hybrid Search and applies                                                                                                                        
      Cross-Encoder joint attention scoring to extract the Top N highest-precision docs.                                                                                                
      """                                                                                                                                                                               
      # 1. Prepare (Query, Document) pairs for the cross-encoder                                                                                                                        
      # Prepending the title gives the reranker vital domain context!                                                                                                                   
      pairs = [                                                                                                                                                                         
          [query, f"{cand['title']}\n{cand['matched_child_text']}"]                                                                                                                     
          for cand in retrieved_candidates
      ]                                                                                                                                                                                 
                                                                                                                                                                                        
      # 2. Predict cross-attention relevance logits                                                                                                                                     
      rerank_scores = reranker.predict(pairs)                                                                                                                                           
                                                                                                                                                                                        
      # 3. Attach rerank scores to candidates                                                                                                                                           
      for i, cand in enumerate(retrieved_candidates):                                                                                                                                   
          cand["rerank_score"] = float(rerank_scores[i])                                                                                                                                
                                                                                                                                                                                        
      # 4. Sort candidates by cross-encoder score descending                                                                                                                            
      reranked_candidates = sorted(retrieved_candidates, key=lambda x: x["rerank_score"], reverse=True)                                                                                 
                                                                                                                                                                                        
      # Re-assign final rank                                                                                                                                                            
      for final_rank, cand in enumerate(reranked_candidates[:top_n], start=1):
          cand["final_rank"] = final_rank
          
      return reranked_candidates[:top_n]

In [ ]:
top_3_reranked = rerank_results(test_query, results, top_n=3)

In [ ]:
top_3_reranked[0]

# Phase: Final LLM Generation

In [ ]:
def format_context_for_llm(reranked_docs: list[dict]) -> str:                                                                                                                         
    """                                                                                                                                                                               
    Assembles the hydrated parent documents into a structured context block                                                                                                           
    with explicit Document IDs for citations.                                                                                                                                         
    """                                                                                                                                                                               
    context_blocks = []                                                                                                                                                               
                                                                                                                                                                                         
    for i, doc in enumerate(reranked_docs, start=1):                                                                                                                                  
        block = (                                                                                                                                                                     
            f"--- DOCUMENT {i} ---\n"                                                                                                                                                 
            f"Doc ID: {doc['doc_id']}\n"                                                                                                                                              
            f"Source: {doc['source_type'].upper()}\n"                                                                                                                                 
            f"Title: {doc['title']}\n"                                                                                                                                                
            f"Content:\n{doc['hydrated_parent_text']}\n"                                                                                                                              
        )                                                                                                                                                                             
        context_blocks.append(block)                                                                                                                                                  
                                                                                                                                                                                         
    return "\n".join(context_blocks)                                                                                                                                                  
                                                                                                                                                                                         
formatted_context = format_context_for_llm(top_3_reranked)                                                                                                                            
print(f"Context assembled! Total characters: {len(formatted_context)}")       

In [ ]:
                                                                                                                                                                                      
SYSTEM_PROMPT = """You are an internal enterprise AI assistant for Redwood Inference.                                                                                                 
    Your task is to answer the user's question based strictly and exclusively on the provided internal documents.                                                                         
                                                                                                                                                                                          
    STRICT OPERATIONAL RULES:                                                                                                                                                             
    1. CITATIONS: Every single factual statement, step, or recommendation you make MUST include an inline citation with its Document ID, formatted as [Doc ID: dsid_xxx].                 
    2. ZERO HALLUCINATION / REFUSAL: If the provided documents do not contain sufficient evidence to answer the question, state: "Information not found in internal documents." Do NOT    
  extrapolate, speculate, or use outside knowledge.                                                                                                                                       
    3. CONFLICTS: If two documents provide contradictory procedures or numbers, explicitly state both viewpoints and cite their respective Document IDs.                                  
    4. TONE: Professional, concise, and structured (use bullet points and bold headers for operational procedures).                                                                       
    """                                                                                                                                                                                   
                                                                                                                                                                                          
def build_rag_prompt(query: str, context: str) -> str:                                                                                                                                
    return f"""{SYSTEM_PROMPT}                                                                                                                                                        
                                                                                                                                                                                          
CONTEXT DOCUMENTS:                                                                                                                                                                    
{context}                                                                                                                                                                             
                                                                                                                                                                                          
USER QUESTION:                                                                                                                                                                        
{query}                                                                                                                                                                               
                                                                                                                                                                                          
ANSWER:"""                

In [ ]:
from transformers import pipeline                                                                                                                                                     
import torch                                                                                                                                                                          
                                                                                                                                                                                          
# 1. Detect GPU device directly without needing the 'accelerate' package                                                                                                              
device = "cuda" if torch.cuda.is_available() else "cpu"                                                                                                                               
print(f"Using compute device: {device}")   

generator = pipeline(                                                                                                                                                                 
        "text-generation",                                                                                                                                                                
        model="Qwen/Qwen2.5-0.5B-Instruct",  # Or "Qwen/Qwen2.5-7B-Instruct" if you have GPU VRAM                                                                                         
        device=device,                                                                                                                                                                
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32                                                                                                     
    )                                                                                                                                                                                     
                                                                                                                                                                                          
# 3. Format the chat prompt
messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXT DOCUMENTS:\n{formatted_context}\n\nUSER QUESTION:\n{test_query}"}
    ]                                                                                                                                                                                
                                                                                                                                                                                          
# 4. Generate grounded answer                                                                                                                                                         
print("Generating grounded answer on GPU...")                                                                                                                                         
output = generator(                                                                                                                                                                   
        messages,                                                                                                                                                                         
        max_new_tokens=450,                                                                                                                                                               
        do_sample=False  # Deterministic greedy decoding for maximum factual accuracy                                                                                                     
    )                                                                                                                                                                                     
  
generated_answer = output[0]["generated_text"][-1]["content"]
print("\n=== FINAL GROUNDED RAG ANSWER ===\n")
print(generated_answer)

# 